# DARPA CADETS — RAG-Based Incident Reporting (v2)

Self-contained multi-window run. Reads only: the CADETS JSON dataset, the Qwen GGUF model, and the ATT&CK STIX bundle. Everything else is built in-notebook.

**Pipeline:** Stage 1 alert generation -> Stage 2 embedding + community detection -> Stage 3 grammar-constrained triple extraction -> Stage 4 KG-anchored RAG over ATT&CK -> Stage 5 per-window evaluation.

Four CADETS Nginx-backdoor windows (W1-W4) with per-window ground-truth IOCs verified against the TA5.1 Ground Truth Report. The email-relay window (sec 4.1) is excluded from scoring (indirect / low host signal) and discussed qualitatively.

## Cell 0 — Reproducibility: seed everything

In [1]:
import os, random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    # best-effort determinism; ignore if backend doesn't support it
    try: torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception: pass
except ImportError:
    pass

print("Seeded with", SEED)

Seeded with 42


## Cell 1 — Stage 1a: parse multi-window CADETS events

Each window carries its own UTC time bounds (report times are EDT = UTC-4) and its own verified IOC set. Events are tagged by which window they fall in (by timestamp) and whether they touch that window's IOCs.

In [2]:
import json, glob
from datetime import datetime, timezone
from collections import Counter

# CADETS E3 is split across three sibling dirs (-official, -official-1, -official-2),
# each with .json, .json.1, ... segments. The wildcard catches all of them so the
# later windows (Apr 11-13) are actually loaded, not just the first dir (Apr 2-6).
CADETS_FILES = sorted(glob.glob(
    "../data/darpa/ta1-cadets-e3-official*/ta1-cadets-e3-official*.json*"
))
P = "com.bbn.tc.schema.avro.cdm18."
print(f"Loading {len(CADETS_FILES)} file(s):")
for p in CADETS_FILES:
    print("  ", p)

def utc_ns(y, mo, d, h, mi):
    return int(datetime(y, mo, d, h, mi, tzinfo=timezone.utc).timestamp() * 1e9)

# Per-window UTC bounds (generous margins) + verified IOCs from the Ground Truth Report.
# Report times are EDT = UTC-4.
# Per-window UTC bounds + verified IOCs + ground-truth ATT&CK technique SETS.
# Report times are EDT = UTC-4.
#
# gt_set is the set of techniques the campaign exhibits in each window, NOT a
# single label. Provenance data is a kill chain: one community legitimately spans
# several techniques. Scoring (Cell 8) counts a community correct if its predicted
# technique's parent is in this set.
#
# !!! PROVISIONAL: every ID below is traceable to the public CADETS Nginx-backdoor
#     narrative (Kairos/Holmes descriptions) AND is reachable by RELATION_TO_TECHNIQUES.
#     RECONCILE each set against your TA5.1 Ground Truth Report before citing.
WINDOWS = {
    "W1": {  # 2018-04-06 11:21-12:08 EDT, sec 3.1
        "start": utc_ns(2018,4,6,14,0), "end": utc_ns(2018,4,6,17,0),
        "ips": {"81.49.200.166","78.205.235.65","200.36.109.214","139.123.0.113",
                "152.111.159.139","154.143.113.18","61.167.39.128"},
        "paths": {"/tmp/vUgefal","/var/log/devc"},
        "gt_set": {"T1190","T1059","T1105","T1071","T1222","T1021","T1055"},
        # T1190 exploit Nginx | T1059 shell exec | T1105 drop vUgefal | T1071 C2
        # T1222 chmod payload | T1021 lateral move | T1055 attempted sshd injection
    },
    "W2": {  # 2018-04-11 15:08-15:15 EDT, sec 3.8 (ends in kernel panic -> thin)
        "start": utc_ns(2018,4,11,18,30), "end": utc_ns(2018,4,11,19,45),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119"},
        "paths": {"sendmail","grain","/tmp/grain"},
        "gt_set": {"T1190","T1059","T1105","T1071"},
    },
    "W3": {  # 2018-04-12 14:00-14:38 EDT, sec 3.13 (richest)
        "start": utc_ns(2018,4,12,17,30), "end": utc_ns(2018,4,12,19,0),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119",
                "53.158.101.118","98.15.44.232","192.113.144.28"},
        "paths": {"tmux-1002","minions","font","XIM","netlog","sendmail","main","/tmp/test"},
        "gt_set": {"T1190","T1059","T1105","T1071","T1222","T1070.004","T1021","T1041"},
        # adds T1070.004 indicator removal (log deletion) | T1041 exfil over C2
    },
    "W4": {  # 2018-04-13 09:04-09:15 EDT, sec 3.14 (ends in crash -> thin)
        "start": utc_ns(2018,4,13,12,30), "end": utc_ns(2018,4,13,14,0),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119","53.158.101.118"},
        "paths": {"pEja72mA","eWq10bVcx","memhelp.so","eraseme","done.so"},
        "gt_set": {"T1190","T1059","T1105","T1070.004","T1055"},
    },
}

GLOBAL_START = min(w["start"] for w in WINDOWS.values())
GLOBAL_END   = max(w["end"]   for w in WINDOWS.values())

def cdm_type(datum):
    return next(iter(datum)).replace(P, "")

# PASS 1: build UUID lookups
subjects, files, netflows = {}, {}, {}
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            t = cdm_type(d); obj = d[P + t]; u = obj.get("uuid")
            if t == "Subject":
                props = obj.get("properties", {}).get("map", {})
                subjects[u] = props.get("exec") or obj.get("cmdLine") or "process"
            elif t == "FileObject":
                files[u] = obj.get("type", "FILE")
            elif t == "NetFlowObject":
                netflows[u] = f'{obj.get("remoteAddress")}:{obj.get("remotePort")}'

print("Subjects:", len(subjects), "Files:", len(files), "NetFlows:", len(netflows))

# PASS 2: collect events in any window, tag with window id
def which_window(ts):
    for wid, w in WINDOWS.items():
        if w["start"] <= ts <= w["end"]:
            return wid
    return None

events = []
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            if cdm_type(d) != "Event": continue
            e = d[P + "Event"]
            ts = e.get("timestampNanos", 0)
            if not (GLOBAL_START <= ts <= GLOBAL_END): continue
            wid = which_window(ts)
            if wid is None: continue
            e["_window"] = wid
            events.append(e)

events.sort(key=lambda e: (e.get("timestampNanos", 0), e.get("uuid", "")))
per_window = dict(Counter(e["_window"] for e in events))
print("Events per window:", per_window)
print("Total events:", len(events))

# Loud guard: a window matching 0 events means missing file coverage or wrong
# time bounds. Fail visibly instead of silently producing an empty eval.
empty = [wid for wid in WINDOWS if per_window.get(wid, 0) == 0]
if empty:
    print(f"\n!! WARNING: these windows matched 0 events: {empty}")
    print("   -> check file coverage (all -official* dirs loaded?) and UTC time bounds.")


Loading 10 file(s):
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.1
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.2
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.3
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.4
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.2
Subjects: 224629 Files: 2305159 NetFlows: 155322
Events per window: {'W1': 376379, 'W2': 174244, 'W3': 318821, 'W4': 212342}
Total events: 1081786


## Cell 1b - DIAGNOSTIC: file/day/window coverage check

Confirms all `-official*` dirs are loaded and that each window's time bounds overlap the data. Run before the heavy cells.

In [3]:
# Cell 1b - DIAGNOSTIC: verify which days/windows are actually covered by the loaded files.
# Run this right after Cell 1, before the heavy downstream cells.
# Single pass over all Event records; no window filtering, no UUID joins.
from datetime import datetime, timezone

def ns_to_str(ts):
    return datetime.fromtimestamp(ts/1e9, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

# 1. What files did the glob actually match?
print("Files matched by glob:")
for p in CADETS_FILES:
    print("  ", p)
print()

# 2. True min/max timestamp across ALL events, plus a per-day histogram.
day_hist = Counter()
ev_min, ev_max, n_ev = None, None, 0
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            if cdm_type(d) != "Event": continue
            ts = d[P+"Event"].get("timestampNanos", 0)
            if ts <= 0: continue
            n_ev += 1
            ev_min = ts if ev_min is None else min(ev_min, ts)
            ev_max = ts if ev_max is None else max(ev_max, ts)
            day = datetime.fromtimestamp(ts/1e9, tz=timezone.utc).strftime("%Y-%m-%d")
            day_hist[day] += 1

print(f"Total events with a timestamp: {n_ev}")
if ev_min:
    print(f"Event time span: {ns_to_str(ev_min)}  ->  {ns_to_str(ev_max)}\n")
    print("Events per calendar day (UTC):")
    for day in sorted(day_hist):
        print(f"   {day}: {day_hist[day]:>8}")
print()

# 3. For each window, does its [start,end] range overlap the data at all?
print(f"{'win':>4} {'start (UTC)':>17} {'end (UTC)':>17}  status")
for wid, w in WINDOWS.items():
    in_data = (ev_min is not None) and not (w["end"] < ev_min or w["start"] > ev_max)
    status = "OK: overlaps data" if in_data else "EMPTY: window is outside the data's time span"
    print(f"{wid:>4} {ns_to_str(w['start']):>17} {ns_to_str(w['end']):>17}  {status}")


Files matched by glob:
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.1
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.2
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.3
   ../data/darpa/ta1-cadets-e3-official-1/ta1-cadets-e3-official-1.json.4
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json
   ../data/darpa/ta1-cadets-e3-official-2/ta1-cadets-e3-official-2.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.1
   ../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json.2

Total events with a timestamp: 41350895
Event time span: 2018-04-02 22:07 UTC  ->  2018-04-13 21:35 UTC

Events per calendar day (UTC):
   2018-04-02:   191264
   2018-04-03:  3167212
   2018-04-04:  3291540
   2018-04-05:  3596535
   2018-04-06:  398597

## Cell 2 — Stage 1b: classify, collapse traffic, render alerts (per window)

Semantic events kept as-is; network traffic collapsed to one channel per (process, IP); plumbing (read/close/mmap/etc.) dropped. Each window sampled at ~10:1 benign:malicious. Alerts carry their window id and label.

In [4]:
SEMANTIC = {"EVENT_EXECUTE","EVENT_WRITE","EVENT_CREATE_OBJECT","EVENT_FORK",
            "EVENT_MODIFY_FILE_ATTRIBUTES","EVENT_MODIFY_PROCESS","EVENT_UNLINK",
            "EVENT_CHANGE_PRINCIPAL","EVENT_RENAME","EVENT_LINK","EVENT_LOGIN",
            "EVENT_MPROTECT","EVENT_TRUNCATE"}
TRAFFIC  = {"EVENT_CONNECT","EVENT_SENDTO","EVENT_RECVFROM","EVENT_ACCEPT",
            "EVENT_SENDMSG","EVENT_RECVMSG","EVENT_BIND"}

def event_targets(e):
    ips, paths = set(), set()
    for k in ("predicateObject","predicateObject2"):
        ref = e.get(k)
        if ref:
            nf = netflows.get(ref.get(P+"UUID"))
            if nf: ips.add(nf.split(":")[0])
    for k in ("predicateObjectPath","predicateObject2Path"):
        pth = e.get(k)
        if pth:
            s = pth.get("string") if isinstance(pth, dict) else pth
            if s: paths.add(s)
    return ips, paths

def is_mal(wid, ips, paths):
    w = WINDOWS[wid]
    if ips & w["ips"]: return True
    return any(any(ioc in p for ioc in w["paths"]) for p in paths)

VERB = {"EVENT_EXECUTE":"executed","EVENT_WRITE":"wrote to","EVENT_CREATE_OBJECT":"created",
        "EVENT_FORK":"forked","EVENT_MODIFY_FILE_ATTRIBUTES":"changed permissions on",
        "EVENT_MODIFY_PROCESS":"modified process","EVENT_UNLINK":"deleted",
        "EVENT_CHANGE_PRINCIPAL":"changed privilege via","EVENT_RENAME":"renamed",
        "EVENT_LINK":"linked","EVENT_LOGIN":"logged in via",
        "EVENT_MPROTECT":"changed memory protection on","EVENT_TRUNCATE":"truncated"}

def role_tag(target):
    """Deterministic semantic hint for a file/host target (helps the embedder)."""
    t = str(target).lower()
    if t in ("a socket/pipe", "<unknown>", ""): return ""
    if "/tmp/" in t or t.startswith("tmp"):      return " (file dropped to temp directory)"
    if "/var/log" in t or "log" in t:            return " (system log file, common anti-forensics target)"
    if t.endswith(".so") or "memhelp" in t:      return " (shared library / loadable module)"
    if "sshd" in t or "ssh" in t:                return " (SSH service process)"
    if "nginx" in t:                             return " (web server process)"
    if any(c.isdigit() for c in t) and "." in t and "/" not in t:
        return " (external network host)"
    return ""

# classify + collapse, separately per window
alerts = []
for wid in WINDOWS:
    sem, chan = [], {}
    for e in events:
        if e["_window"] != wid: continue
        et = e.get("type","?")
        ips, paths = event_targets(e)
        mal = is_mal(wid, ips, paths)
        subj_ref = e.get("subject") or {}
        eprops = (e.get("properties") or {}).get("map") or {}
        proc = eprops.get("exec") or subjects.get(subj_ref.get(P+"UUID")) or "process"
        if et in TRAFFIC:
            ip = next(iter(ips), None)
            if ip:
                key = (proc, ip)
                if key not in chan:
                    chan[key] = {"proc":proc,"ip":ip,"count":0,"mal":mal,"ts":e.get("timestampNanos")}
                chan[key]["count"] += 1
        elif et in SEMANTIC:
            tgt = next(iter(paths), None) or next(iter(ips), None) or "a socket/pipe"
            sem.append({"type":et,"proc":proc,"target":tgt,"mal":mal,"ts":e.get("timestampNanos")})

    mal_sem  = [x for x in sem if x["mal"]]
    ben_sem  = [x for x in sem if not x["mal"]]
    mal_chan = [v for v in chan.values() if v["mal"]]
    ben_chan = [v for v in chan.values() if not v["mal"]]
    n_mal = len(mal_sem) + len(mal_chan)
    ben_keep      = random.sample(ben_sem,  min(len(ben_sem),  max(n_mal*10, 10)))
    ben_chan_keep = random.sample(ben_chan, min(len(ben_chan), max(n_mal, 5)))

    for x in mal_sem + ben_keep:
        alerts.append({"window":wid,"label":"malicious" if x["mal"] else "benign","ts":x["ts"],
                       "text":f'Process {x["proc"]} {VERB.get(x["type"],x["type"])} {x["target"]}{role_tag(x["target"])}'})
    for c in mal_chan + ben_chan_keep:
        alerts.append({"window":wid,"label":"malicious" if c["mal"] else "benign","ts":c["ts"],
                       "text":f'Process {c["proc"]} connected to {c["ip"]} ({c["count"]} packets)'})

    print(f"{wid}: {n_mal} malicious ({len(mal_sem)} sem + {len(mal_chan)} chan), "
          f"{len(ben_keep)+len(ben_chan_keep)} benign kept")

alerts.sort(key=lambda a:(a["window"], a["ts"]))
texts  = [a["text"]  for a in alerts]
labels = [a["label"] for a in alerts]
awins  = [a["window"] for a in alerts]
print("\nTotal alerts:", len(alerts))
print("Sample malicious alerts:")
for a in [a for a in alerts if a["label"]=="malicious"][:15]:
    print(f'  [{a["window"]}] {a["text"]}')


W1: 13 malicious (8 sem + 5 chan), 143 benign kept
W2: 6 malicious (3 sem + 3 chan), 66 benign kept
W3: 49 malicious (44 sem + 5 chan), 507 benign kept
W4: 20 malicious (14 sem + 6 chan), 217 benign kept

Total alerts: 1021
Sample malicious alerts:
  [W1] Process nginx connected to 78.205.235.65 (302 packets)
  [W1] Process nginx connected to 81.49.200.166 (6 packets)
  [W1] Process nginx wrote to <unknown>
  [W1] Process nginx wrote to <unknown>
  [W1] Process nginx connected to 200.36.109.214 (89 packets)
  [W1] Process nginx wrote to /tmp/vUgefal (file dropped to temp directory)
  [W1] Process nginx changed permissions on /tmp/vUgefal (file dropped to temp directory)
  [W1] Process master executed /tmp/vUgefal (file dropped to temp directory)
  [W1] Process vUgefal connected to 139.123.0.113 (157 packets)
  [W1] Process nginx deleted /tmp/vUgefal (file dropped to temp directory)
  [W1] Process vUgefal connected to 61.167.39.128 (2 packets)
  [W1] Process vUgefal changed permissions 

## Cell 3 — Stage 2: embed + community detection

Same model as the primary CIC-IDS pipeline (all-MiniLM-L6-v2). Community detection at a fixed threshold; communities scored for purity using the per-alert label.

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(texts, convert_to_tensor=True, show_progress_bar=True)

THRESHOLD = 0.50
communities = util.community_detection(emb, threshold=THRESHOLD, min_community_size=3)

print(f"{len(communities)} communities at threshold {THRESHOLD}\n")
for i, comm in enumerate(communities):
    cl = [labels[j] for j in comm]
    n_mal = cl.count("malicious")
    purity = max(n_mal, len(comm)-n_mal)/len(comm)
    dom = "MALICIOUS" if n_mal > len(comm)/2 else "benign"
    win_mix = Counter(awins[j] for j in comm).most_common(1)[0][0]
    print(f"C{i}: {len(comm)} alerts | {n_mal} mal | purity {purity:.2f} | {dom} | mainly {win_mix}")
    if dom == "MALICIOUS":
        for j in comm:
            if labels[j]=="malicious":
                print(f"      [{awins[j]}] {texts[j]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

/home/amenadiel/anaconda3/envs/thesis/lib/python3.11/site-packages/sentence_transformers/util/retrieval.py:298: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  cos_scores = embeddings[start_idx : start_idx + batch_size] @ embeddings.T


35 communities at threshold 0.5

C0: 411 alerts | 0 mal | purity 1.00 | benign | mainly W3
C1: 94 alerts | 6 mal | purity 0.94 | benign | mainly W4
C2: 77 alerts | 0 mal | purity 1.00 | benign | mainly W3
C3: 62 alerts | 6 mal | purity 0.90 | benign | mainly W3
C4: 48 alerts | 0 mal | purity 1.00 | benign | mainly W3
C5: 41 alerts | 0 mal | purity 1.00 | benign | mainly W1
C6: 30 alerts | 6 mal | purity 0.80 | benign | mainly W3
C7: 24 alerts | 1 mal | purity 0.96 | benign | mainly W3
C8: 24 alerts | 21 mal | purity 0.88 | MALICIOUS | mainly W3
      [W3] Process nginx wrote to minions
      [W2] Process nginx wrote to grain
      [W4] Process nginx wrote to pEja72mA
      [W3] Process nginx wrote to font
      [W3] Process nginx wrote to XIM
      [W4] Process nginx wrote to eWq10bVcx
      [W3] Process nginx wrote to tmux-1002
      [W1] Process nginx connected to 200.36.109.214 (89 packets)
      [W1] Process nginx wrote to /tmp/vUgefal (file dropped to temp directory)
      [W3] Pr

## Cell 4 — threshold sweep (reproducibility-friendly diagnostic)

Shows purity vs. coverage across thresholds. Coverage (malicious alerts landing in any malicious-dominant community) is the more stable metric than community count.

In [6]:
def summarize(threshold, min_size=3):
    comms = util.community_detection(emb, threshold=threshold, min_community_size=min_size)
    mal_doms, purities, covered = 0, [], set()
    for comm in comms:
        cl = [labels[j] for j in comm]
        n_mal = cl.count("malicious")
        purities.append(max(n_mal, len(comm)-n_mal)/len(comm))
        if n_mal > len(comm)/2:
            mal_doms += 1
            for j in comm:
                if labels[j]=="malicious": covered.add(j)
    tot = sum(1 for l in labels if l=="malicious")
    ap = sum(purities)/len(purities) if purities else 0
    return len(comms), mal_doms, len(covered), tot, ap

print(f"{'thresh':>7} {'#comm':>6} {'#mal-dom':>9} {'mal-cov':>9} {'avg-purity':>11}")
for t in [0.45,0.50,0.55,0.60,0.65,0.70]:
    n,md_,cov,tot,ap = summarize(t)
    print(f"{t:>7.2f} {n:>6} {md_:>9} {cov:>3}/{tot:<5} {ap:>11.2f}")

 thresh  #comm  #mal-dom   mal-cov  avg-purity
   0.45     31         6  43/88           0.89
   0.50     35         8  48/88           0.90
   0.55     34         6  46/88           0.89
   0.60     47        10  54/88           0.96
   0.65     43         9  48/88           0.96
   0.70     51         8  48/88           0.97


## Cell 5 — Stage 3 setup: load model + grammar + relations

Open subject/target, relation constrained to an enum. CIC-IDS relations kept unchanged; host-provenance relations appended. Relation->technique bridge is a superset of the CIC-IDS one.

In [7]:
import re
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = "../models/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=-1, seed=SEED, verbose=False)
print("Model loaded.")

def normalise_entity(text):
    text = re.sub(r'[^a-z0-9_]+','_', text.lower().strip())
    return re.sub(r'_+','_', text).strip('_')[:80]

SECURITY_RELATIONS = ['PERFORMS_RECONNAISSANCE','PERFORMS_PORT_SCAN','BRUTE_FORCES_CREDENTIAL',
    'ACCESS_CREDENTIALS','EXPLOITS_VULNERABILITY','ESTABLISHES_C2','PERFORMS_BEACONING',
    'CAUSES_DENIAL_OF_SERVICE','MOVES_LATERALLY','EXFILTRATES_DATA','EXECUTES_PAYLOAD']
# Network-flow relations (PORT_SCAN / RECONNAISSANCE / DoS / BEACONING) describe
# packet-level behaviour that simply is not observable in host audit data; leaving
# them in the grammar invites coerced hallucinations (e.g. sshd "PERFORMS_PORT_SCAN").
# Restrict the host enum to relations a provenance event can actually evidence.
NETWORK_ONLY = {'PERFORMS_RECONNAISSANCE','PERFORMS_PORT_SCAN','PERFORMS_BEACONING',
                'CAUSES_DENIAL_OF_SERVICE','BRUTE_FORCES_CREDENTIAL'}
HOST_RELATIONS = [r for r in SECURITY_RELATIONS if r not in NETWORK_ONLY] + \
    ['CHANGES_PERMISSIONS','DELETES_FILE','INJECTS_PROCESS','WRITES_FILE']

RELATION_TO_TECHNIQUES = {
    'PERFORMS_RECONNAISSANCE':['T1046','T1595','T1590'], 'PERFORMS_PORT_SCAN':['T1046','T1595'],
    'BRUTE_FORCES_CREDENTIAL':['T1110','T1110.001','T1110.003'], 'ACCESS_CREDENTIALS':['T1555','T1078','T1110'],
    'EXPLOITS_VULNERABILITY':['T1190','T1203'], 'ESTABLISHES_C2':['T1071','T1071.001','T1071.004'],
    'PERFORMS_BEACONING':['T1071','T1071.004'], 'CAUSES_DENIAL_OF_SERVICE':['T1498','T1499','T1499.001'],
    'MOVES_LATERALLY':['T1021','T1570'], 'EXFILTRATES_DATA':['T1041','T1048'],
    'EXECUTES_PAYLOAD':['T1059','T1059.007','T1105'],
    'CHANGES_PERMISSIONS':['T1222','T1548'], 'DELETES_FILE':['T1070.004','T1070'],
    'INJECTS_PROCESS':['T1055','T1055.001'], 'WRITES_FILE':['T1105'],
}

TRIPLE_SCHEMA = {'type':'object','properties':{'triples':{'type':'array','minItems':1,'maxItems':4,
    'items':{'type':'object','properties':{'subject':{'type':'string'},
        'relation':{'type':'string','enum':HOST_RELATIONS},'target':{'type':'string'}},
    'required':['subject','relation','target']}}},'required':['triples']}
host_grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))

RELATION_GUIDE = """Relation definitions (use exactly as written):
  EXPLOITS_VULNERABILITY : initial exploit of a service (malformed request to a web server)
  ESTABLISHES_C2         : outbound connection to an external command-and-control address
  EXECUTES_PAYLOAD       : running a dropped binary or command
  WRITES_FILE            : writing/dropping a file to disk
  CHANGES_PERMISSIONS    : changing file permissions (often to enable execution/elevation)
  INJECTS_PROCESS        : injecting code into another process
  DELETES_FILE           : removing a file (often to cover tracks)
  PERFORMS_PORT_SCAN     : probing multiple ports/hosts on the network
  EXFILTRATES_DATA       : transferring data off the host"""
print("Grammar + bridge ready —", len(HOST_RELATIONS), "relations")

llama_context: n_ctx_seq (4096) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


Model loaded.
Grammar + bridge ready — 10 relations


## Cell 6 — Stage 3: extract triples per malicious-dominant community

In [8]:
def extract_triples(alert_texts, tag):
    block = "\n".join(f"- {t}" for t in alert_texts[:6])
    assert "malicious" not in block and "benign" not in block, f"label leak {tag}"
    prompt = f"""[INST] You are a cybersecurity analyst analyzing host audit events.
Extract 1-4 semantic triples describing the attack behaviour.

{RELATION_GUIDE}

subject = process/actor; relation = the ONE best fit; target = file/address/process.
Name what you observe. No generic placeholders.

Example:
{{"triples": [
  {{"subject":"nginx","relation":"EXPLOITS_VULNERABILITY","target":"web_server_process"}},
  {{"subject":"nginx","relation":"ESTABLISHES_C2","target":"external_c2_address"}},
  {{"subject":"nginx","relation":"WRITES_FILE","target":"tmp_implant_binary"}}
]}}

ALERTS ({tag}):
{block}

Return ONLY valid JSON. [/INST]"""
    out = llm(prompt, max_tokens=512, temperature=0, seed=SEED,
              grammar=host_grammar, repeat_penalty=1.1, stop=["[/INST]"])
    try: triples = json.loads(out["choices"][0]["text"].strip()).get("triples", [])
    except Exception as e:
        print(f"  [{tag}] parse failed: {e}"); triples = []
    valid, seen_fallback = [], 0
    for t in triples:
        s = normalise_entity(str(t.get("subject",""))); r = str(t.get("relation","")).upper().strip()
        o = normalise_entity(str(t.get("target","")))
        if not s or not o: continue
        if r not in HOST_RELATIONS: r = "EXECUTES_PAYLOAD"; seen_fallback += 1
        valid.append({"subject":s,"relation":r,"target":o})
    return valid, seen_fallback

def malicious_dominant(comms):
    return [i for i,c in enumerate(comms)
            if sum(1 for j in c if labels[j]=="malicious") > len(c)/2]

MAL_COMMS = malicious_dominant(communities)
print("Malicious-dominant communities:", MAL_COMMS, "\n")

community_triples, fallback_counts = {}, {}
for cid in MAL_COMMS:
    member_texts = [texts[j] for j in communities[cid]]
    win = Counter(awins[j] for j in communities[cid]).most_common(1)[0][0]
    tr, fb = extract_triples(member_texts, f"C{cid}")
    community_triples[cid] = {"triples":tr, "window":win}
    fallback_counts[cid] = fb
    print(f"C{cid} (mainly {win}):")
    for t in tr:
        print(f"   ({t['subject']}, {t['relation']}, {t['target']})")
    if fb: print(f"   [coerced to fallback: {fb}]")
    print()

Malicious-dominant communities: [8, 18, 23, 24, 25, 27, 32, 34] 

C8 (mainly W3):
   (nginx, WRITES_FILE, minions)
   (nginx, WRITES_FILE, grain)
   (nginx, WRITES_FILE, peja72ma)
   (nginx, WRITES_FILE, font)

C18 (mainly W3):
   (nginx, WRITES_FILE, tmp_grain)
   (nginx, CHANGES_PERMISSIONS, tmp_grain)
   (nginx, WRITES_FILE, tmp_ewq10bvcx)
   (nginx, CHANGES_PERMISSIONS, tmp_ewq10bvcx)

C23 (mainly W3):
   (xim, WRITES_FILE, var_log_sendmail)
   (xim, DELETES_FILE, var_log_sendmail)
   (sendmail, ESTABLISHES_C2, 127_0_0_1)
   (xim, CHANGES_PERMISSIONS, var_log_sendmail)

C24 (mainly W1):
   (xim, ESTABLISHES_C2, 53_158_101_118)
   (test, ESTABLISHES_C2, 192_113_144_28)
   (vugefal, ESTABLISHES_C2, 61_167_39_128)

C25 (mainly W3):
   (cron, EXECUTES_PAYLOAD, libexec_ld_elf_so_1)
   (sshd, WRITES_FILE, tmp_font)

C27 (mainly W4):
   (peja72ma, ESTABLISHES_C2, 53_158_101_118)
   (peja72ma, WRITES_FILE, done_so)
   (peja72ma, WRITES_FILE, eraseme)
   (peja72ma, WRITES_FILE, memhelp_so)


## Cell 7 — Stage 4 setup: build ATT&CK embeddings from STIX (no saved index)

Parses enterprise-attack.json fresh and embeds technique name+description in memory. Replaces the ChromaDB dependency entirely.

In [9]:
import numpy as np
from sentence_transformers import util as sutil

def load_attck(path="../data/attck/enterprise-attack.json"):
    bundle = json.load(open(path))
    techs = {}
    for obj in bundle["objects"]:
        if obj.get("type")!="attack-pattern" or obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue
        tid = next((r["external_id"] for r in obj.get("external_references",[])
                    if r.get("source_name")=="mitre-attack"), None)
        if not tid: continue
        tac = [p["phase_name"] for p in obj.get("kill_chain_phases",[])
               if p.get("kill_chain_name")=="mitre-attack"]
        techs[tid] = {"name":obj.get("name",""), "description":obj.get("description",""),
                      "tactic":tac[0] if tac else ""}
    return techs

attck = load_attck()
tech_ids = list(attck.keys())
tech_texts = [f"{attck[t]['name']}. {attck[t]['description']}" for t in tech_ids]
tech_embs = model.encode(tech_texts, convert_to_tensor=True, show_progress_bar=True)
print(f"Embedded {len(tech_ids)} ATT&CK techniques")

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Embedded 691 ATT&CK techniques


## Cell 8 — Stage 5: KG-anchored retrieval + per-window evaluation

Ground truth is per window (all four CADETS Nginx windows = T1190, exploit public-facing application). Candidates come from each community's dominant relation; ranked by alert-text similarity. Scored as parent-match.

In [10]:
TOP_K = 5

def in_gt_set(gt_set, pred):
    """Correct if the predicted technique's parent matches any technique in the
    window's ground-truth set (parent-level, so subtechniques count)."""
    if not pred or pred == "Unknown": return False
    pred_parent = pred.split(".")[0]
    return any(pred_parent == g.split(".")[0] for g in gt_set)

def retrieve(query, k=TOP_K):
    q = model.encode(query, convert_to_tensor=True)
    sims = sutil.pytorch_cos_sim(q, tech_embs)[0].cpu().numpy()
    return [tech_ids[i] for i in np.argsort(-sims)[:k]]

def kg_candidates(triples):
    rels = [t["relation"] for t in triples]
    if not rels: return []
    dom = Counter(rels).most_common(1)[0][0]
    return [t for t in RELATION_TO_TECHNIQUES.get(dom,[]) if t in attck]

results = []
for cid in MAL_COMMS:
    info = community_triples[cid]
    triples, win = info["triples"], info["window"]
    gt_set = WINDOWS[win]["gt_set"]
    alert_text = " ".join(texts[j] for j in communities[cid])
    triple_text = " ".join(f"{t['subject']} {t['relation']} {t['target']}" for t in triples)
    cands = kg_candidates(triples)
    retrieved = retrieve(alert_text[:400] + " " + triple_text)
    if cands:
        q_emb = model.encode(alert_text[:400], convert_to_tensor=True)
        c_embs = tech_embs[[tech_ids.index(t) for t in cands]]
        sims = sutil.pytorch_cos_sim(q_emb, c_embs)[0].cpu().tolist()
        pred = cands[int(np.argmax(sims))]
    else:
        pred = retrieved[0] if retrieved else "Unknown"
    dom_rel = Counter(t["relation"] for t in triples).most_common(1)[0][0] if triples else "none"
    results.append({"cid":cid,"window":win,"gt_set":sorted(gt_set),"pred":pred,"dom_rel":dom_rel,
                    "retr_hit":bool(set(retrieved) & gt_set),"parent":in_gt_set(gt_set,pred)})

print(f"{'C':>3} {'win':>4} {'pred':>9} {'dom_rel':>22} {'retr':>5} {'hit':>5}  gt_set")
for r in results:
    print(f"{r['cid']:>3} {r['window']:>4} {r['pred']:>9} {r['dom_rel']:>22} "
          f"{str(r['retr_hit']):>5} {str(r['parent']):>5}  {','.join(r['gt_set'])}")

# per-window summary
print("\nPer-window parent-match:")
for wid in WINDOWS:
    wr = [r for r in results if r["window"]==wid]
    if wr:
        h = sum(r["parent"] for r in wr)
        print(f"  {wid}: {h}/{len(wr)} communities correct")
    else:
        print(f"  {wid}: no malicious-dominant community formed")

hits = sum(r["parent"] for r in results)
print(f"\nOverall: {hits}/{len(results)} = {hits/max(len(results),1):.1%}")

/home/amenadiel/anaconda3/envs/thesis/lib/python3.11/site-packages/sentence_transformers/util/similarity.py:45: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  return torch.mm(a_norm, b_norm.transpose(0, 1)).to_dense()


  C  win      pred                dom_rel  retr   hit  gt_set
  8   W3     T1105            WRITES_FILE False  True  T1021,T1041,T1059,T1070.004,T1071,T1105,T1190,T1222
 18   W3     T1105            WRITES_FILE  True  True  T1021,T1041,T1059,T1070.004,T1071,T1105,T1190,T1222
 23   W3     T1105            WRITES_FILE False  True  T1021,T1041,T1059,T1070.004,T1071,T1105,T1190,T1222
 24   W1     T1071         ESTABLISHES_C2  True  True  T1021,T1055,T1059,T1071,T1105,T1190,T1222
 25   W3     T1059       EXECUTES_PAYLOAD False  True  T1021,T1041,T1059,T1070.004,T1071,T1105,T1190,T1222
 27   W4     T1105            WRITES_FILE False  True  T1055,T1059,T1070.004,T1105,T1190
 32   W3 T1070.004           DELETES_FILE  True  True  T1021,T1041,T1059,T1070.004,T1071,T1105,T1190,T1222
 34   W3 T1070.004           DELETES_FILE  True  True  T1021,T1041,T1059,T1070.004,T1071,T1105,T1190,T1222

Per-window parent-match:
  W1: 1/1 communities correct
  W2: no malicious-dominant community formed
  W3: 6/6

## Cell 9 — save run state (for reuse / write-up)

Saves alerts, communities, triples, and results. Embeddings saved separately so reload skips the 11GB parse.

In [11]:
import torch
state = {
    "seed": SEED, "threshold": THRESHOLD,
    "alerts": alerts,
    "communities": [[int(j) for j in c] for c in communities],
    "mal_comms": MAL_COMMS,
    "community_triples": {str(k):v for k,v in community_triples.items()},
    "fallback_counts": {str(k):v for k,v in fallback_counts.items()},
    "results": results,
}
with open("../data/darpa/cadets_v2_run.json","w") as f:
    json.dump(state, f, indent=2)
torch.save(emb, "../data/darpa/cadets_v2_emb.pt")
print("Saved run state + embeddings.")

Saved run state + embeddings.
